In [ ]:
!pip install agentpy

In [ ]:
import agentpy as ap
import numpy as np
import random, json
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import seaborn as sns, IPython
from matplotlib import pyplot as plt, cm

#######################################
# CLASS FOR AGENT
#######################################
class MazeAgent(ap.Agent):
    '''
    Initializing agent elements:
    - actions: 4 possible actions
    - env: reference to its environment
    '''
    def setup(self):
        # Actions are linked to a movement in the grid.
        self.actions = {'up': (-1,0), 'down': (1, 0), 'left': (0, -1), 'right': (0, 1)}
        self.env = self.model.env

    '''
    Actual action execution. Each step the agent performs a
    '''
    def execute(self):
        action = self.choose_action()
        self.env.move_by(self, self.actions[action])
        self.reward += self.env.get_reward(self.get_position())
        return action

    '''
    Get position of agent in environment
    '''
    def get_position(self):
        return self.env.positions[self]

    '''
    Dumb agent chooses a random action
    '''
    def choose_action(self):
        return random.choice(list(self.actions.keys()))



#######################################
# CLASS FOR ENVIRONMENT
#######################################
class Maze(ap.Grid):
    def setup(self):
        # Initialize the maze environment
        self.rewards = self.p.maze[:, :]

    '''
    Reward function. The returned value is used to update Q-values
    '''
    def get_reward(self, state):
        reward = self.rewards[state]
        if self.rewards[state] != 0 and self.rewards[state] != -1:
            self.rewards[state] = 0
        return reward


#######################################
# CLASS FOR THE SYSTEM (Ag, Env)
#######################################
class MazeModel(ap.Model):
    def setup(self):
        self.env = Maze(self, shape=maze.shape)
        self.agent = MazeAgent(self)
        self.env.add_agents([self.agent], positions=[self.p.init])
        self.agent.reward = 0

    def step(self):
        self.agent.execute()

    def update(self):
        # If agent reaches the goal, simulation stops
        if self.agent.get_position() == self.model.p.goal:
            print('ending')
            self.stop()




####################################################
# EXECUTION AND VISUALIZATION OF SPECIFIC INSTANCES
####################################################
def animation_plot(model, ax):
    n, m = model.p.maze.shape
    grid = np.zeros((n, m))
    grid[model.p.maze == -1] = -1
    grid[model.p.goal] = 2

    color_dict = {0:'#ffffff', -1:'#000000', 3:'#0000ff', 2:'#00ff00', 1:'#ffff00'}
    ap.gridplot(grid, ax=ax, color_dict=color_dict, convert=True)
    agent = list(model.env.agents)[0]
    grid[model.env.positions[agent]] = 3
    ap.gridplot(grid, ax=ax, color_dict=color_dict, convert=True)
    ax.set_title("Maze Agent\nWall hits: {}".format(-agent.reward))


# Reading information from numpy file
maze = -np.load('maze_example.npy')
n = len(maze)

parameters = {
    'maze': maze,
    'init': (n - 2, n - 3),
    'goal': (8, 10),
    'steps': 100
}

fig = plt.figure(figsize=(7,7))
ax = fig.add_subplot(111)
mazeModel = MazeModel(parameters)
animation = ap.animate(mazeModel, fig, ax, animation_plot)
IPython.display.HTML(animation.to_jshtml())

ending
